# 6. 卷积神经网络


## 6.1 从全连接层到卷积

> **为什么处理图像不能直接用多层感知机（MLP），而必须发明卷积？**

### 1. MLP 处理图像的“三座大山”

假设我们有一张 $1000 \times 1000$ 像素的彩色图片（3 通道）：
1.  **参数爆炸**：如果隐藏层只有 1000 个神经元，全连接层的权重矩阵大小为 $(1000 \times 1000 \times 3) \times 1000 = 3 \times 10^9$。这需要 **12GB** 显存来存储仅仅一层的权重，根本无法训练。
2.  **空间结构丢失**：MLP 要求把图片展平（Flatten）成一维。这导致像素点 $(i, j)$ 与其邻居 $(i+1, j)$ 的关系，和它与远处像素 $(i+500, j)$ 的关系在模型看来是一样的。
3.  **平移脆弱性**：如果在训练集里猫都在左上角，而测试集里的猫出现在右下角，MLP 必须重新学习一遍“右下角的猫”，它无法自动识别这种位置变化。

---

### 2. 卷积的两个物理学直觉

为了解决上述问题，科学家提出了两个核心假设：

#### 2.1. 平移不变性 (Translation Invariance)

**直觉**：不管“猫”出现在图片的哪个位置，它的特征（如尖耳朵、胡须）应该是相同的。

**数学结论**：我们的“特征检测器”在图片的任何地方都应该是**同一套权重**。

#### 2.2. 局部性 (Locality)

**直觉**：要识别一个“鼻子”，我只需要看鼻子周围的像素，不需要看图片另一角的“尾巴”。

**数学结论**：检测器每次只需要关注一个**很小的窗口**。

---

### 3. 数学演化：从全连接到卷积

#### 3.1. 全连接层公式
$$h_{i,j} = \sum_{k,l} W_{i,j,k,l} x_{k,l}$$
这里 $W$ 是一个四维张量。每一对输出坐标 $(i, j)$ 都要看遍输入坐标 $(k, l)$。

#### 3.2. 引入“平移不变性”
我们假设权重只取决于输出点 $(i, j)$ 与输入点 $(k, l)$ 的**相对距离** $(a, b)$，其中 $a = i-k, b = j-l$。
$$h_{i,j} = \sum_{a,b} V_{a,b} x_{i+a, j+b}$$
**变化**：$W$ 变成了 $V$。不管输出 $(i, j)$ 在哪，权重 $V$ 是一样的。**参数量从 $O(H^2 W^2)$ 降到了 $O(HW)$。**

#### 3.3. 引入“局部性”
我们假设当相对距离 $a$ 或 $b$ 超过一定范围（比如 3 像素）时，权重 $V_{a,b} = 0$。
$$h_{i,j} = \sum_{a=-\Delta}^{\Delta} \sum_{b=-\Delta}^{\Delta} V_{a,b} x_{i+a, j+b}$$
**变化**：求和范围缩小到一个极小的窗口。**这就是卷积！** $V$ 就是我们说的**卷积核（Kernel）**。

---

### 4. 对比 MLP 和 CNN 的参数量差异。



In [2]:
from torch import nn

def compare_parameter_counts(height: int, weight: int, channel: int) -> None:
    """对比全连接层和卷积层的参数量。
    
    Args:
        height: 图像高度。
        weight: 图像宽度。
        channel: 输入通道数。
    """
    input_dim = height * weight * channel
    hidden_dim = 1000

    # 1. 模拟全连接层
    fc = nn.Linear(input_dim, hidden_dim)
    fc_params = sum(p.numel() for p in fc.parameters())

    # 2. 模拟卷积层 (假设 3 * 3 卷积核，1000个输出通道)
    # 虽然通道数一样，但卷积核是局部共享的
    conv = nn.Conv2d(in_channels=channel, out_channels=hidden_dim, kernel_size=3)
    conv_params = sum(p.numel() for p in conv.parameters())

    print(f"图像尺寸: {height}x{weight}, 通道: {channel}")
    print(f"全连接层参数量: {fc_params:,}")
    print(f"卷积层参数量: {conv_params:,}") # 3 * 3 * 3 * 1000 + 1000
    print(f"参数压缩倍数: {fc_params / conv_params:.2f}x")

# 运行对比 (假设 128x128 的小图)
compare_parameter_counts(128, 128, 3)

图像尺寸: 128x128, 通道: 3
全连接层参数量: 49,153,000
卷积层参数量: 28,000
参数压缩倍数: 1755.46x


**实验结论**：
你会发现，对于 $128 \times 128$ 的图片，MLP 需要近 **5000 万**个参数，而卷积层只需要 **2.8 万**个。卷积层不仅效率高，而且强制模型学习“局部特征”。

---

### 5. 核心术语总结

*   **通道 (Channels)**：图像的维度。输入通常是 RGB（3通道），隐藏层可以有成百上千个通道（每个通道代表一种特征，如颜色、边缘、纹理）。
*   **感受野 (Receptive Field)**：输出中的一个像素能“看”到输入图片的范围。层数越深，感受野越大。
*   **互相关 (Cross-Correlation)**：代码里实际运行的算法，直接把卷积核 $K$ 放在 $X$ 的窗口上，对应位置相乘再求和。
*   **卷积 (Convolution)**：在进行乘法之前，先将卷积核 $K$ 水平翻转，再垂直翻转（相当于旋转 180 度），然后再执行相乘求和。但深度学习中由于核是训练出来的，翻不翻转结果一样，所以通用术语叫卷积。

---


## 6.2 图像卷积

> 从底层实现二维互相关运算，并学习如何通过数据“学习”出一个卷积核。

### 1. 二维互相关运算

在深度学习框架中，“卷积层”实际上实现的是互相关运算。

#### 1.1 运算逻辑
卷积核（Kernel）在输入张量上滑动。在每个位置，输入子张量与卷积核按元素相乘并求和，得到输出张量中的单点数值。

#### 1.2 输出形状公式
若输入形状为 $n_h \times n_w$，卷积核形状为 $k_h \times k_w$，则输出形状为：
$$(n_h - k_h + 1) \times (n_w - k_w + 1)$$

---

### 2. 手动实现卷积算子

我们将编写一个通用的互相关函数，这是所有 CNN 层的数学心脏。


In [4]:
import torch
from torch import nn, Tensor

def corr2d(X: Tensor, K: Tensor) -> Tensor:
    """计算二维互相关运算。
    
    这是卷积层最底层的数学实现。

    Args:
        X: 输入张量，形状为 (h, w)。
        K: 卷积核张量，形状为 (kh, kw)。
    
    Returns:
        Tensor: 输出特征图，形状为 (h - kh + 1, w - kw + 1)。
    """
    h, w = X.shape
    kh, kw = K.shape

    # 1. 根据公式预分配输出空间
    Y = torch.zeros((h - kh + 1, w - kw + 1), device=X.device)

    # 2. 嵌套循环实现滑动窗口
    # 注意：在生产环境（如 nn.Conv2d）中，这部分由高效的 C++/CUDA 算子完成
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            # 核心动作：切片 (Slicing) -> 逐元素相乘 -> 求和
            Y[i, j] = (X[i: i + kh, j: j+kw] * K).sum()

    return Y

# --- 简单验证 ---
X_test = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K_test = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
# 输出应为 [[19, 25], [37, 43]]
# 计算过程示例: (0*0 + 1*1 + 3*2 + 4*3) = 19
print(f"验证输出:\n{corr2d(X_test, K_test)}")

验证输出:
tensor([[19., 25.],
        [37., 43.]])


---

### 3. 自定义卷积层类


In [11]:
class MyConv2D(nn.Module):
    """自定义二维卷积层模块。"""

    def __init__(self, kernel_size: tuple[int, int]):
        """
        Args:
            kernel_size: 卷积核的形状 (高度, 宽度)。
        """
        super().__init__()
        # 注册可学习参数：权重 (Weight) 和 偏置 (Bias)
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, X: Tensor) -> Tensor:
        """前向传播逻辑。"""
        return corr2d(X, self.weight) + self.bias
    
# 1. 实例化自定义卷积层，传入 2x2 的卷积核尺寸
conv_layer = MyConv2D(kernel_size=(2, 2))

# 2. 准备输入数据
X_test = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])

# 3. 执行前向传播
Y_test = conv_layer(X_test)

# 4. 打印结果
print(list(conv_layer.parameters()))
print(f"输入形状: {X_test.shape}")
print(f"输出形状: {Y_test.shape}")
print(f"前向传播结果:\n{Y_test}")

[Parameter containing:
tensor([[0.9993, 0.6915],
        [0.3089, 0.0335]], requires_grad=True), Parameter containing:
tensor([0.], requires_grad=True)]
输入形状: torch.Size([3, 3])
输出形状: torch.Size([2, 2])
前向传播结果:
tensor([[1.7522, 3.7855],
        [7.8520, 9.8852]], grad_fn=<AddBackward0>)


---

### 4. 手工设计边缘检测器
